# Task 1: Model Training and Optimization Pipeline
Data preprocessing, hyperparameter tuning via Cross-Validation (Grid Search, Random Search, Bayesian Optimization), trackio experiment tracking, and final evaluation on the test set.

In [2]:
!pip install optuna trackio

  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached mako-1.3.10-py3-none-any.whl.metadata (2.9 kB)
Using cached optuna-4.8.0-py3-none-any.whl (419 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 20.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 21.2 MB/s  0:00:00eta 0:00:01
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)
Using cached semantic_version-2.10.0-py2.py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import numpy as np
import pickle
import time
import os
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import trackio
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

print('All imports successful.')

All imports successful.


## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`, encode categorical variables with `LabelEncoder`, and save encoders for use in the Streamlit app.

In [4]:
train_df = pd.read_csv('Dataset/train.csv')
test_df  = pd.read_csv('Dataset/test.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('\nColumn dtypes:\n', train_df.dtypes)
print('\nMissing values (train):\n', train_df.isnull().sum())

Train shape: (11128, 15)
Test shape : (2782, 15)

Column dtypes:
 location                 str
city                     str
latitude             float64
longitude            float64
price                  int64
numBathrooms           int64
numBalconies           int64
isNegotiable           int64
SecurityDeposit        int64
Status                   str
Size_ft²               int64
BHK                    int64
rooms_num              int64
property_type            str
verification_days    float64
dtype: object

Missing values (train):
 location             0
city                 0
latitude             0
longitude            0
price                0
numBathrooms         0
numBalconies         0
isNegotiable         0
SecurityDeposit      0
Status               0
Size_ft²             0
BHK                  0
rooms_num            0
property_type        0
verification_days    0
dtype: int64


In [5]:
# Fill missing values
train_df['verification_days'].fillna(train_df['verification_days'].median(), inplace=True)
test_df['verification_days'].fillna(train_df['verification_days'].median(), inplace=True)

# Categorical columns to encode
CATEGORICAL_COLS = ['location', 'city', 'Status', 'property_type']

encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    # Fit on combined train+test values to avoid unseen label issues
    all_vals = pd.concat([train_df[col], test_df[col]]).astype(str)
    le.fit(all_vals)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col]  = le.transform(test_df[col].astype(str))
    encoders[col] = le
    print(f'Encoded "{col}" -> {len(le.classes_)} classes')

# Save encoders
with open('models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print('\nEncoders saved to models/encoders.pkl')

# Feature / target split
TARGET = 'price'
FEATURES = [c for c in train_df.columns if c != TARGET]

X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values
X_test  = test_df[FEATURES].values
y_test  = test_df[TARGET].values

print(f'\nFeatures ({len(FEATURES)}): {FEATURES}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

Encoded "location" -> 702 classes
Encoded "city" -> 4 classes
Encoded "Status" -> 3 classes
Encoded "property_type" -> 6 classes

Encoders saved to models/encoders.pkl

Features (14): ['location', 'city', 'latitude', 'longitude', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Status', 'Size_ft²', 'BHK', 'rooms_num', 'property_type', 'verification_days']
X_train: (11128, 14), X_test: (2782, 14)


## 2. Hyperparameter Tuning using Cross-Validation

We compare three strategies on the same 60-trial budget using 5-fold CV:
1. **Grid Search** – exhaustive over exactly 60 combinations
2. **Random Search** – 60 random draws from the full range
3. **Bayesian Optimization** – 60 trials with Optuna TPE sampler

In [ ]:
# Initialize trackio
trackio.init(project='UrbanNest-RentPrediction')
print('trackio initialised.')

* Created new run: brave-forest-1
trackio initialised.


In [8]:
# ── 1. Grid Search ──────────────────────────────────────────────────────────
grid_param_grid = {
    'n_estimators'    : [50, 100, 150, 200],    # 4 values
    'max_depth'       : [10, 15, 20, 25, 30],   # 5 values
    'min_samples_split': [2, 5, 8]              # 3 values  → 4×5×3 = 60
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print('Running Grid Search (60 combinations × 5 folds) …')
t0 = time.time()
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    grid_param_grid,
    cv=kf,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

grid_best_score = -grid_search.best_score_
grid_best_params = grid_search.best_params_

# Build per-trial best-so-far curve
grid_results = grid_search.cv_results_
grid_mean_errors = -grid_results['mean_test_score']
grid_best_so_far = np.minimum.accumulate(grid_mean_errors)

print(f'  Best CV MAE : {grid_best_score:,.0f} INR')
print(f'  Best params : {grid_best_params}')
print(f'  Time taken  : {grid_time:.1f}s')

trackio.log({
    'method'          : 'Grid Search',
    'n_iterations'    : int(60),
    'time_seconds'    : float(round(grid_time, 2)),
    'best_cv_mae'     : float(round(grid_best_score, 2)),
    'best_n_estimators'    : int(grid_best_params['n_estimators']),
    'best_max_depth'       : int(grid_best_params['max_depth']),
    'best_min_samples_split': int(grid_best_params['min_samples_split']),
})

Running Grid Search (60 combinations × 5 folds) …
  Best CV MAE : 13,501 INR
  Best params : {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 200}
  Time taken  : 181.6s


In [9]:
# ── 2. Random Search ─────────────────────────────────────────────────────────
from scipy.stats import randint

random_param_dist = {
    'n_estimators'    : randint(50, 201),
    'max_depth'       : randint(10, 31),
    'min_samples_split': randint(2, 11)
}

print('Running Random Search (60 iterations × 5 folds) …')
t0 = time.time()
random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    random_param_dist,
    n_iter=60,
    cv=kf,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42,
    verbose=0
)
random_search.fit(X_train, y_train)
random_time = time.time() - t0

random_best_score  = -random_search.best_score_
random_best_params = random_search.best_params_

random_results   = random_search.cv_results_
random_mean_errors = -random_results['mean_test_score']
random_best_so_far = np.minimum.accumulate(random_mean_errors)

print(f'  Best CV MAE : {random_best_score:,.0f} INR')
print(f'  Best params : {random_best_params}')
print(f'  Time taken  : {random_time:.1f}s')

trackio.log({
    'method'          : 'Random Search',
    'n_iterations'    : 60,
    'time_seconds'    : float(round(random_time, 2)),
    'best_cv_mae'     : float(round(random_best_score, 2)),
    'best_n_estimators'    : random_best_params['n_estimators'],
    'best_max_depth'       : random_best_params['max_depth'],
    'best_min_samples_split': random_best_params['min_samples_split'],
})

Running Random Search (60 iterations × 5 folds) …
  Best CV MAE : 13,532 INR
  Best params : {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}
  Time taken  : 188.1s


In [10]:
# ── 3. Bayesian Optimization (Optuna) ────────────────────────────────────────
bayes_trial_errors = []  # MAE for each trial (not best-so-far yet)

def optuna_objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 50, 200),
        'max_depth'       : trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state'    : 42,
    }
    model = RandomForestRegressor(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=kf,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    mae = -scores.mean()
    bayes_trial_errors.append(mae)
    return mae

print('Running Bayesian Optimization (60 trials via Optuna TPE) …')
t0 = time.time()
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_objective, n_trials=60, show_progress_bar=False)
bayes_time = time.time() - t0

bayes_best_score  = study.best_value
bayes_best_params = study.best_params
bayes_best_so_far = np.minimum.accumulate(bayes_trial_errors)

print(f'  Best CV MAE : {bayes_best_score:,.0f} INR')
print(f'  Best params : {bayes_best_params}')
print(f'  Time taken  : {bayes_time:.1f}s')

trackio.log({
    'method'          : 'Bayesian Optimization (Optuna)',
    'n_iterations'    : 60,
    'time_seconds'    : float(round(bayes_time, 2)),
    'best_cv_mae'     : float(round(bayes_best_score, 2)),
    'best_n_estimators'    : bayes_best_params['n_estimators'],
    'best_max_depth'       : bayes_best_params['max_depth'],
    'best_min_samples_split': bayes_best_params['min_samples_split'],
})

Running Bayesian Optimization (60 trials via Optuna TPE) …
  Best CV MAE : 13,495 INR
  Best params : {'n_estimators': 155, 'max_depth': 24, 'min_samples_split': 2}
  Time taken  : 342.0s


## 3. Evaluation & Plots

In [11]:
# ── Plot 1: Trials vs Best CV Error ──────────────────────────────────────────
trials = np.arange(1, 61)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(trials, grid_best_so_far,   label='Grid Search',   color='steelblue',   linewidth=2)
ax.plot(trials, random_best_so_far, label='Random Search', color='darkorange',  linewidth=2, linestyle='--')
ax.plot(trials, bayes_best_so_far,  label='Bayesian (Optuna)', color='forestgreen', linewidth=2, linestyle='-.')

ax.set_xlabel('Number of Trials / Iterations', fontsize=13)
ax.set_ylabel('Best CV MAE (INR)', fontsize=13)
ax.set_title('Hyperparameter Optimization: Compute Budget vs. Best Error\n(UrbanNest Rent Prediction)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('plots/trials_vs_error.png', dpi=150)
plt.show()
print('Saved: plots/trials_vs_error.png')

Saved: plots/trials_vs_error.png


In [12]:
# ── Plot 2: Optuna Hyperparameter Space ───────────────────────────────────────
# Scatter plot of all 60 Optuna trials coloured by MAE
trial_df = study.trials_dataframe()
sc = plt.figure(figsize=(12, 4))

ax1 = sc.add_subplot(1, 3, 1)
scatter = ax1.scatter(trial_df['params_n_estimators'], trial_df['params_max_depth'],
                      c=trial_df['value'], cmap='viridis_r', alpha=0.8, s=60)
plt.colorbar(scatter, ax=ax1, label='CV MAE (INR)')
ax1.set_xlabel('n_estimators'); ax1.set_ylabel('max_depth')
ax1.set_title('n_estimators vs max_depth')

ax2 = sc.add_subplot(1, 3, 2)
scatter2 = ax2.scatter(trial_df['params_n_estimators'], trial_df['params_min_samples_split'],
                       c=trial_df['value'], cmap='viridis_r', alpha=0.8, s=60)
plt.colorbar(scatter2, ax=ax2, label='CV MAE (INR)')
ax2.set_xlabel('n_estimators'); ax2.set_ylabel('min_samples_split')
ax2.set_title('n_estimators vs min_samples_split')

ax3 = sc.add_subplot(1, 3, 3)
scatter3 = ax3.scatter(trial_df['params_max_depth'], trial_df['params_min_samples_split'],
                       c=trial_df['value'], cmap='viridis_r', alpha=0.8, s=60)
plt.colorbar(scatter3, ax=ax3, label='CV MAE (INR)')
ax3.set_xlabel('max_depth'); ax3.set_ylabel('min_samples_split')
ax3.set_title('max_depth vs min_samples_split')

sc.suptitle('Optuna Bayesian Search — Hyperparameter Space Exploration\n(darker = lower MAE = better)', fontsize=13)
plt.tight_layout()
plt.savefig('plots/optuna_hyperparameter_space.png', dpi=150)
plt.show()
print('Saved: plots/optuna_hyperparameter_space.png')

Saved: plots/optuna_hyperparameter_space.png


## 4. Final Testing & Model Saving
Report best hyperparameters, train the best overall model on full `train.csv`, evaluate on `test.csv`, and save the model.

In [13]:
# ── Report Best Hyperparameters ───────────────────────────────────────────────
print('=' * 60)
print('BEST HYPERPARAMETERS BY METHOD')
print('=' * 60)

print(f'\n1. Grid Search')
print(f'   Best CV MAE : {grid_best_score:,.0f} INR')
print(f'   Params      : {grid_best_params}')

print(f'\n2. Random Search')
print(f'   Best CV MAE : {random_best_score:,.0f} INR')
print(f'   Params      : {random_best_params}')

print(f'\n3. Bayesian Optimization (Optuna)')
print(f'   Best CV MAE : {bayes_best_score:,.0f} INR')
print(f'   Params      : {bayes_best_params}')

# Identify overall winner
scores = {
    'Grid Search'   : (grid_best_score,   grid_best_params),
    'Random Search' : (random_best_score, random_best_params),
    'Bayesian (Optuna)': (bayes_best_score, bayes_best_params),
}
best_method = min(scores, key=lambda k: scores[k][0])
best_cv_mae, best_params = scores[best_method]

print(f'\n>>> WINNER: {best_method} with CV MAE = {best_cv_mae:,.0f} INR')
print(f'    Best hyperparameters: {best_params}')

BEST HYPERPARAMETERS BY METHOD

1. Grid Search
   Best CV MAE : 13,501 INR
   Params      : {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 200}

2. Random Search
   Best CV MAE : 13,532 INR
   Params      : {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}

3. Bayesian Optimization (Optuna)
   Best CV MAE : 13,495 INR
   Params      : {'n_estimators': 155, 'max_depth': 24, 'min_samples_split': 2}

>>> WINNER: Bayesian (Optuna) with CV MAE = 13,495 INR
    Best hyperparameters: {'n_estimators': 155, 'max_depth': 24, 'min_samples_split': 2}


In [15]:
# ── Train Best Model on Full Train Set ───────────────────────────────────────
# Strip random_state key if it crept in from Optuna params
clean_params = {k: v for k, v in best_params.items() if k != 'random_state'}

best_model = RandomForestRegressor(**clean_params, random_state=42)
best_model.fit(X_train, y_train)

# Evaluate on test set
y_pred  = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)

print(f'Final Test MAE ({best_method}): {test_mae:,.0f} INR')

trackio.log({
    'final_test_mae'   : float(round(test_mae, 2)),
    'winning_method'   : best_method,
    'final_n_estimators'    : clean_params.get('n_estimators'),
    'final_max_depth'       : clean_params.get('max_depth'),
    'final_min_samples_split': clean_params.get('min_samples_split'),
})

# Save model and feature list
with open('models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('models/feature_names.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

print('Saved: models/best_rf_model.pkl')
print('Saved: models/feature_names.pkl')
print('Saved: models/encoders.pkl  (saved earlier in cell 1)')

Final Test MAE (Bayesian (Optuna)): 12,458 INR
Saved: models/best_rf_model.pkl
Saved: models/feature_names.pkl
Saved: models/encoders.pkl  (saved earlier in cell 1)


In [16]:
# ── Summary Table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Method'       : ['Grid Search', 'Random Search', 'Bayesian (Optuna)'],
    'Best CV MAE'  : [f'{grid_best_score:,.0f}', f'{random_best_score:,.0f}', f'{bayes_best_score:,.0f}'],
    'Time (s)'     : [f'{grid_time:.1f}', f'{random_time:.1f}', f'{bayes_time:.1f}'],
    'n_estimators' : [grid_best_params['n_estimators'], random_best_params['n_estimators'], bayes_best_params['n_estimators']],
    'max_depth'    : [grid_best_params['max_depth'], random_best_params['max_depth'], bayes_best_params['max_depth']],
    'min_samples_split': [grid_best_params['min_samples_split'], random_best_params['min_samples_split'], bayes_best_params['min_samples_split']],
})
print(summary.to_string(index=False))
print(f'\nFinal Test MAE: {test_mae:,.0f} INR')

           Method Best CV MAE Time (s)  n_estimators  max_depth  min_samples_split
      Grid Search      13,501    181.6           200         30                  2
    Random Search      13,532    188.1           138         22                  2
Bayesian (Optuna)      13,495    342.0           155         24                  2

Final Test MAE: 12,458 INR
